# 10 · 병렬 알고리즘 cuda-cccl (`cuda.compute`)

> **CuPy 2일 집중 코스 — Day 2 / 단원 7 (병렬 알고리즘, GTC 05 기반)**

08·09에서 커널을 **직접** 짰다면, 여기서는 NVIDIA가 제공하는 **검증된 병렬 알고리즘**으로
같은 일을 더 쉽고 빠르게 합니다. `cuda.compute`(cuda-cccl)의 reduce·scan·transform·sort 등은
손으로 튜닝한 CUDA 커널 수준의 성능을 Python에서 제공합니다.

## 이 노트북의 위치 (07 개념과의 관계)
- 07~09의 개념(인덱싱·coalescing·공유메모리·atomic)을 **알고리즘이 내부에서 처리** → 우리는 *무엇을* 할지만 지정
- 직접 커널(생산성↓·성능↑·제어↑) vs cccl(생산성↑·검증된 성능) 의 트레이드오프

### 조금 더 구체적으로: CCCL이 실제로 하는 일

**CCCL**(CUDA Core Compute Libraries)은 NVIDIA가 오랫동안 유지해온 세 C++ 템플릿 라이브러리 —
**Thrust**(STL과 유사한 고수준 병렬 알고리즘), **CUB**(블록/워프 단위 저수준 최적화 프리미티브),
**libcudacxx**(표준 C++ 라이브러리의 CUDA 대응) — 를 하나의 우산 아래 묶은 프로젝트입니다.
`cuda.compute`는 이 중 Thrust/CUB가 오래전부터 제공해온 reduce·scan·sort·transform 같은
**정형화된 병렬 패턴**을 파이썬에서 그대로 호출할 수 있게 감싼 실험적 바인딩입니다.

왜 이것이 "검증된 성능"인가: 08~09에서 우리가 직접 짠 커널은 특정 GPU·특정 문제 크기에 맞춰서만
튜닝됩니다. 반면 CCCL의 알고리즘은 NVIDIA 엔지니어들이 **컴파일 타임에 아키텍처(compute
capability)별 타일 크기·워프 활용 전략·메모리 접근 패턴을 특수화(specialize)** 해둔 코드라서,
Ampere든 Hopper든 세대에 맞는 최적 경로가 자동으로 선택됩니다. 07의 `ReductionKernel`이 만드는
단순한 트리 리덕션을, 2단계 블록 리덕션·워프 셔플 명령 등으로 훨씬 정교하게 구현한 버전이라고
이해하면 됩니다.

성능과 제어력의 위치를 표로 정리하면:

| | 직접 커널 (08·09·11) | cuda-cccl (10) | CuPy 고수준 (07, `cp.sum` 등) |
|---|---|---|---|
| 제어 | 최대 | 정형 패턴 + 커스텀 연산 | 최소(내부에 위임) |
| 생산성 | 최소 | 균형 | 최대 |
| 성능 튜닝 부담 | 사용자 몫 | NVIDIA가 아키텍처별로 선튜닝 | CuPy가 알아서 |

## 학습 목표
- `reduce_into`로 합/커스텀 리덕션을 수행한다.
- 사용자 정의 이항연산·`unary_transform`·iterator를 활용한다.
- 언제 cccl / Numba / CuPy를 쓸지 판단한다.

## 목차
1. [cuda-cccl 이란 & 설치](#1)
2. [reduce_into — 합](#2)
3. [커스텀 이항연산 리덕션](#3)
4. [unary_transform](#4)
5. [Iterators (메모리 없는 시퀀스)](#5)
6. [언제 무엇을 쓰나](#6)
7. [스캔·커스텀·평가](#8) · 8. [연습](#7-ex)

> `cuda-cccl`은 **실험적 패키지**(2026 기준 `cuda.compute`, v0.5.x)입니다. API가 버전에 따라 바뀔 수 있습니다.

<a id="1"></a>
## 1. cuda-cccl 이란 & 설치

**CCCL**(CUDA Core Compute Libraries)의 Python 인터페이스 `cuda.compute` 는 배열/범위 단위 **병렬 알고리즘**을 제공합니다.
- reduce, scan, sort, transform, … (아키텍처 이식성 + 고성능)

설치(강의장 환경): `pip install "cuda-python[cu13]" "cuda-cccl[cu13]"`. 미설치 시 아래 셀은 안내만 출력합니다.

### 조금 더 구체적으로

- **JIT 컴파일 비용**: `op`으로 넘긴 파이썬 함수(뒤에 나올 `add_even`, `max_op` 등)는 최초 호출 시
  CUDA 커널로 컴파일합니다. 00에서 다룬 "일회성 오버헤드"와 같은 현상이 여기서도
  발생하므로, 성능을 재고 싶다면 반드시 워밍업 후 측정해야 합니다(이 노트북의 예제는 정확성 검증이
  목적이라 별도 `bench` 호출은 생략했지만, 실전에서 성능을 비교할 때는 `course_utils.bench`로 감싸세요).
- **왜 실험적(experimental)인가**: `cuda.compute`는 아직 API가 빠르게 바뀌는 초기 단계 패키지입니다.
  강의장 환경마다 설치가 안 되어 있을 수 있어, 이 노트북의 모든 예제 셀은 `HAS_CCCL` 플래그로
  방어적으로 건너뛸 수 있게 작성되어 있습니다(바로 다음 셀 참고).

In [ ]:
import numpy as np, cupy as cp
from course_utils import print_env, allclose
print_env()
try:
    import cuda.compute
    from cuda.compute import OpKind, CountingIterator, TransformIterator
    HAS_CCCL = True
    print('cuda.compute 사용 가능')
except Exception as e:
    HAS_CCCL = False
    print('cuda-cccl 미설치 — `pip install "cuda-python[cu13]" "cuda-cccl[cu13]"` 후 실행하세요. (',e,')')

<a id="2"></a>
## 2. reduce_into — 합

`reduce_into(d_in, d_out, op, num_items, h_init)`. `op`은 내장 `OpKind.PLUS` 등.
`h_init`은 **host(numpy) 초기값 배열**, `d_out`은 결과를 담을 device 배열.

### 조금 더 구체적으로

`reduce_into`는 개념적으로 07에서 만든 `cp.ReductionKernel`과 같은 **map-reduce 트리 리덕션**을
수행하지만, 사용자가 CUDA C 문자열(`map_expr`, `reduce_expr`)을 쓰는 대신 **내장 연산 열거형
(`OpKind`)** 하나만 지정하면 됩니다.

- **`h_init`이 host 배열인 이유**: CCCL은 리덕션 연산의 항등원(identity element, 예: `PLUS`의
  항등원은 0)을 커널 컴파일 시점에 상수로 반영하기 위해 host 메모리에서 값을 읽습니다. 이는 CuPy
  `ReductionKernel`의 다섯 번째 인자 `identity`(예: `'0.0f'`)와 정확히 같은 역할입니다.
- **`d_out`이 왜 배열(스칼라 아님)인가**: 00에서 CuPy의 리덕션 결과가 0차원 배열로 돌아온다고
  배웠던 것과 같은 이유입니다 — device 메모리에 결과를 남겨 불필요한 동기화를 피합니다. `d_out`은
  최소 크기 1의 device 배열을 미리 할당해 넘깁니다.
- **성능 감각**: 100만 개 `int64` 원소 합산은 원소당 8바이트 → 총 8 MB를 한 번 읽는 전형적인
  메모리 바운드 연산입니다. 최신 GPU의 메모리 대역폭(수백 GB/s~수 TB/s)을 기준으로 하면 이론상
  마이크로초 단위로 끝나야 하는 작업이며, `reduce_into`는 이런 대역폭에 근접하도록 이미 튜닝되어
  있습니다. 직접 짠 순진한 커널은 여러 번의 커널 실행과 atomic 경쟁 때문에 이보다 느리기 쉽습니다
  — `09_numba_histogram`에서 겪은 atomic 경쟁 이슈를 떠올려보세요.
- CuPy로 표준 합계만 필요하면 `d_in.sum()` 한 줄로 충분합니다 — `reduce_into`가 진짜 필요한
  이유는 다음 절의 **커스텀 연산**입니다.

In [ ]:
dtype = np.int64
d_in  = cp.arange(1, 1_000_001, dtype=dtype)   # 1..1,000,000
d_out = cp.empty(1, dtype=dtype)
h_init = np.array([0], dtype=dtype)

cuda.compute.reduce_into(
    d_in=d_in, 
    d_out=d_out, 
    op=OpKind.PLUS, 
    num_items=d_in.size, 
    h_init=h_init
)

print('cccl 합:', int(d_out[0]), '/ cupy 합:', int(d_in.sum()))
allclose(int(d_in.sum()), int(d_out[0]), name='reduce sum')

<a id="3"></a>
## 3. 커스텀 이항연산 리덕션

내장 연산 대신 **함수**를 넘기면 임의의 리덕션이 됩니다(람다 불가). 예: 짝수만 합산.

### 조금 더 구체적으로

07의 `ReductionKernel`은 `map_expr`(원소별 전처리)과 `reduce_expr`(두 값을 합치는 결합 연산)을
**분리**해서 받습니다 — 예를 들어 L1 노름은 map이 `fabsf(x)`, reduce가 `a+b`였습니다. 아래
`add_even` 예제는 이 둘을 **하나의 이항 함수**로 합친 형태로 볼 수 있습니다: 짝수 여부 판단
(map에 해당하는 필터링)과 덧셈(reduce)을 한 파이썬 함수 안에서 함께 처리합니다.

중요한 제약 한 가지: 병렬 리덕션은 원소들을 **어떤 순서·어떤 트리 구조로 묶어도** 같은 결과가
나와야 하므로, `op`은 반드시 **결합법칙(associative)** 을 만족해야 합니다(교환법칙까지 만족하면
더 안전합니다). `add_even`은 두 인자를 대칭적으로 처리하므로 안전합니다. 반면 `a - b`처럼 순서에
의존하는 연산을 커스텀 op으로 넘기면, 스레드/블록마다 다른 결합 순서로 계산되어 **실행할 때마다
다른(틀린) 결과**가 나올 수 있습니다 — 08·09에서 강조한 "워프·블록의 실행 순서는 보장되지 않는다"는
사실이 여기서도 그대로 적용됩니다.

내부 동작: 커스텀 `op`으로 넘긴 파이썬 함수는 CUDA 디바이스 함수로 즉석 컴파일되어,
CUB의 리덕션 템플릿 안에 끼워 넣습니다. C++ 템플릿의 타입 파라미터 자리에 우리가 만든 파이썬
함수가 대입된다고 생각하면 이해가 쉽습니다.

CUDA 환경에 따라 잘 안될 경우 아래 명령 실행

conda install --override-channels -c conda-forge cuda-nvcc cuda-nvrtc -y

> 커스텀 `op`(위에서 만들 `add_even` 같은 파이썬 함수)은 CUDA 커널로
> 컴파일됩니다. 이 컴파일에는 NVIDIA의 `nvrtc`(런타임 컴파일러)와 경우에 따라 `nvcc`가 필요한데,
> conda로 CuPy만 설치하고 이 두 패키지가 빠져 있으면 **커스텀 연산을 쓰는 셀에서만** 컴파일
> 에러가 날 수 있습니다(내장 `OpKind`만 쓰는 앞 절의 `reduce_into`는 영향받지 않습니다). 그럴 때
> 위 명령으로 컴파일러를 보충 설치하세요.

In [ ]:
def add_even(a, b):
    return (a if a % 2 == 0 else 0) + (b if b % 2 == 0 else 0)

d_in = cp.array([1, 2, 3, 4, 5, 6], dtype=np.int32)
d_out = cp.empty(1, dtype=np.int32)
h_init = np.array([0], dtype=np.int32)

cuda.compute.reduce_into(
    d_in=d_in, 
    d_out=d_out, 
    op=add_even,           # 우리가 만든 커스텀 함수를 전달
    num_items=d_in.size, 
    h_init=h_init
)

print('짝수 합:', int(d_out[0]))   # 2+4+6 = 12


<a id="4"></a>
## 4. unary_transform

각 원소에 함수를 적용해 출력 배열에 씁니다: `unary_transform(d_in, d_out, func, num_items)`.

### 조금 더 구체적으로

`unary_transform`은 07의 `cp.ElementwiseKernel`과 같은 문제(원소별 함수 적용)를 풀지만, C 코드
문자열 대신 **파이썬 함수**를 그대로 받는다는 점이 다릅니다. 07의 `clamp_scale` 예제처럼 조건문이
들어간 로직도 파이썬 함수로 자연스럽게 표현할 수 있습니다.

성능 관점에서는 둘 다 결과적으로 "각 스레드가 원소 하나를 담당"하는 동일한 실행 패턴(2절의
SPMD)으로 컴파일되므로, 단순 원소별 연산이라면 어느 쪽을 써도 큰 차이가 없습니다. 선택 기준은
성능이 아니라 **어떤 API가 지금 상황에 더 자연스러운가**입니다 — 이미 파이썬 함수로 로직이
있다면 `unary_transform`, C 스타일 삼항연산자·`raw` 인덱싱이 필요하면 `ElementwiseKernel`
(07의 연습 B 참고)이 더 편합니다.

`d_out`을 `d_in`과 같은 배열로 넘기면 in-place 변환도 가능하지만(버전에 따라 지원 여부가 다를 수
있음), 이 노트북 예제처럼 별도 출력 배열을 쓰는 편이 더 안전하고 명확합니다.

In [ ]:
def sq(x): 
    return x * x

d_in = cp.arange(10, dtype=np.int32)
d_out = cp.empty(10, dtype=np.int32)

cuda.compute.unary_transform(
    d_in=d_in, 
    d_out=d_out, 
    op=sq, 
    num_items=d_in.size
)

print(cp.asnumpy(d_out))   # [ 0  1  4  9 16 25 36 49 64 81]

<a id="5"></a>
## 5. Iterators — 메모리 없는 시퀀스

`CountingIterator`/`TransformIterator`는 **메모리를 할당하지 않고** 시퀀스를 표현해 알고리즘에 바로 넣습니다.
예: 10부터의 정수 100개의 합(배열 생성 없이).

### 조금 더 구체적으로

**메모리 절약을 숫자로 보면**: `int32` 100만 개를 `cp.arange`로 만들면 4 MB의 VRAM을 실제로
할당하고 채웁니다(00에서 다룬 host↔device 전송과는 별개로, 이건 device 내부 할당·초기화
비용입니다). 반면 `CountingIterator`는 시작값 하나만 저장한 '수식'이며, 리덕션 커널이 스레드마다
`시작값 + 전역 인덱스`를 그 자리에서 계산합니다 — 08에서 `cuda.grid(1)`로 전역 인덱스를 구해
배열에 접근하던 패턴과 본질적으로 같은 계산을, 아예 **배열 자체를 없애고** 인덱스 계산만 남긴
것입니다. 데이터가 수억~수십억 개 규모로 커지면 이 차이(GB 단위 할당 vs 8바이트 상태)가 실질적인
메모리 예산을 결정합니다.

**`TransformIterator`**(맨 위 import에는 포함되어 있지만 이 노트북에서 별도 예제는 생략)는
`CountingIterator`에 한 겹 더해, "인덱스를 계산 → 그 값에 함수를 적용"까지 메모리 할당 없이
파이프라인으로 연결합니다. 예를 들어 `TransformIterator(CountingIterator(0), sq)`는
`[0, 1, 4, 9, ...]`라는 배열을 실제로 만들지 않고도 `reduce_into`에 바로 넣어 제곱합을 구할 수
있습니다 — `unary_transform`으로 배열을 만든 뒤 `reduce_into`로 리덕션하는 **두 단계(중간 배열
필요)** 대신, **한 번의 커널 실행 + 0바이트 중간 저장**으로 끝나는 것이 핵심 이점입니다. 이런
반복자 합성(iterator composition)은 C++ Thrust에서 `thrust::transform_iterator`로 오래전부터
쓰이던 패턴을 그대로 파이썬으로 가져온 것입니다.

In [ ]:
# 시작값을 10으로 하는 가상 이터레이터 생성
first = CountingIterator(np.int32(10)) 
# 이 기능의 진정한 매력은 GPU 메모리를 단 1바이트도 쓰지 않는다는 점입니다.
# cp.arange로 배열을 만들면 GPU VRAM을 차지하지만, CountingIterator는 숫자가 필요할 때마다 GPU 코어가 즉석에서 계산해 내므로 대용량 데이터를 처리할 때 메모리 절약 효과가 큼
d_out = cp.empty(1, dtype=np.int32)

cuda.compute.reduce_into(
    d_in=first, 
    d_out=d_out, 
    op=OpKind.PLUS, 
    num_items=100, 
    h_init=np.array([0], dtype=np.int32)
)

# 10부터 109까지 총 100개 숫자의 합 계산 (정답: 5950)
print('합(10..109):', int(d_out[0]))

<a id="6"></a>
## 6. 언제 무엇을 쓰나

| 선택 | 쓸 때 |
|------|-------|
| **CuPy 함수**(`cp.sum`…) | 표준 연산, 가장 간단 |
| **`@cupy.fuse`/Elementwise/Reduction** | 원소·리덕션 융합, 약간의 커스텀 |
| **cuda-cccl**(`cuda.compute`) | reduce/scan/sort/transform + **커스텀 연산**을 검증된 성능으로 |
| **Numba CUDA** | 공유메모리·atomic·복잡한 인덱싱 등 **완전한 제어** |

### 조금 더 구체적으로

이 판단 기준을 07의 매핑표(개념 → 노트북)와 나란히 놓으면 이렇게 이어집니다: **CuPy 표준
함수**는 이미 존재하는 조합(mean, sum, sort 등)에 최적, **Elementwise/Reduction/`@cupy.fuse`**는
07에서 봤듯 C 식 하나로 표현되는 단순 커스텀 연산에 적합, **cuda-cccl**은 reduce/scan/sort/
transform 같은 **정형 병렬 패턴이지만 연산 자체는 우리 마음대로**(임의의 결합 함수, 커스텀
구조체) 바꾸고 싶을 때, **Numba CUDA**(08·09)는 공유메모리·atomic·비정형 인덱싱처럼 ** 새로운 알고리즘**을 짤 때 고려합니다.

<a id="7"></a>
## 7. 스캔 · 커스텀 타입 · 종합 평가

리덕션 외에 **스캔(prefix)**, **커스텀 구조체 타입** 평가 과제를 다룹니다.
> 스캔 API는 실험적이라 버전에 따라 인자 순서가 다를 수 있습니다 — 공식 예제로 확인하세요.

### 조금 더 구체적으로

**스캔(scan)** 은 리덕션 다음으로 중요한 병렬 프리미티브입니다. 순차적으로 보면 각 출력이 이전
출력에 의존하는 것처럼 보이지만(`out[i] = out[i-1] + in[i-1]`), 실제로는 **Hillis-Steele**나
**Blelloch(work-efficient)** 같은 트리 기반 병렬 알고리즘으로 `O(log n)` 단계 안에 계산할 수
있습니다 — 이는 08~09에서 본 트리 리덕션의 확장판이라고 볼 수 있습니다. 스캔이 병렬 컴퓨팅에서
중요한 이유는 정렬(radix sort), 스트림 압축(조건을 만족하는 원소만 골라내기), 희소 자료구조
인덱싱 등 **표면적으로 순차적으로 보이는 문제들을 병렬화하는 만능 도구**이기 때문입니다.

이어지는 7.1~7.3은 스캔 API 사용법, 커스텀 구조체 타입(`gpu_struct`)으로의 확장, 그리고 실전
문제(센서 데이터의 '직전 최고치')를 통해 지금까지 배운 reduce/scan/커스텀 op을 종합합니다.

### 7.1 스캔(prefix sum)

**exclusive scan**: `out[i] = op(in[0..i-1])` (자기 앞까지 누적). 누적합·러닝 통계의 기본.
-  Exclusive Scan (배타적 누적합)
 * 배열이 온통 1로 채워져 있는데 결과가 0, 1, 2, 3...으로 나오는 이유는 자기 자신을 포함하지 않고(Exclusive) 이전까지의 값들만 더하기 때문입니다.
 * 인덱스 0: 이전 값이 없으므로 초기값인 0
 * 인덱스 1: 이전 값(인덱스 0의 1) 하나만 있으므로 0 + 1 = 1
 * 인덱스 2: 이전 값(인덱스 0, 1의 1) 두 개가 있으므로 0 + 1 + 1 = 2
 * 병렬 컴퓨팅에서 각 스레드가 메모리 어디에 데이터를 써야 할지 오프셋(인덱스 시작 위치)을 계산할 때 사용되는 알고리즘

### 조금 더 구체적으로: inclusive vs exclusive

CCCL은 **inclusive scan**(`out[i] = op(in[0..i])`, 자기 자신 포함)도 별도로 제공합니다
(`inclusive_scan`, 이 노트북에서는 다루지 않음). 언제 어느 쪽을 쓰는지는 문제가 정하는 것이지
임의 선택이 아닙니다 — 예를 들어 "정렬 후 각 원소가 출력 배열의 어느 위치에 써야 하는가"를 구하는
**오프셋 계산**에는 반드시 **exclusive**가 필요합니다(자기 자신의 개수까지 포함하면 위치가 한
칸씩 밀리기 때문). 반대로 "지금까지의 누적 합계"를 그대로 보여주고 싶다면 inclusive가 더
자연스럽습니다.

`init_value`(`h_init`)의 역할은 앞서 2절 `reduce_into`의 `h_init`과 동일합니다 — 리덕션 연산의
항등원을 지정하는 것입니다. `PLUS`라면 0, 7.3에서 쓸 `max` 연산이라면 "그 dtype에서 가능한
가장 작은 값"이 항등원 역할을 합니다.

In [ ]:
# 러닝 합 (exclusive scan, PLUS)
N = 10
d_in = cp.ones(N, dtype=np.int32)
d_out = cp.empty(N, dtype=np.int32)
h_init = np.array([0], dtype=np.int32)

cuda.compute.exclusive_scan(
    d_in=d_in, 
    d_out=d_out, 
    op=OpKind.PLUS, 
    init_value=h_init,
    num_items=N
)
print(cp.asnumpy(d_out))   # [0 1 2 3 4 5 6 7 8 9]


### 7.2 커스텀 구조체 타입 (`gpu_struct`)

사용자 정의 타입으로도 리덕션이 됩니다. 예: 픽셀들 중 **녹색(g)이 최대**인 픽셀 찾기(공식 예제).

### 조금 더 구체적으로

`@gpu_struct`로 선언한 `Pixel`은 내부적으로 numpy의 구조체 dtype(레코드 타입)에 대응되게 컴파일됩니다.
`cp.random.randint(...).view(Pixel.dtype)`는 **메모리를 복사
하지 않고** `(N, 3)` int32 배열을 `N`개의 `Pixel` 레코드 배열로 **재해석(reinterpret)** 하는
연산입니다 — 00에서 다룬 CuPy의 뷰(view) vs 복사(copy) 개념이 여기서도 그대로 적용됩니다
(`.view()`가 성립하려면 메모리 레이아웃이 정확히 맞아야 하며, r/g/b 세 개의 int32가 연속으로
있는 배열 구조가 `Pixel{r,g,b}` 구조체와 바이트 단위로 일치하기 때문에 가능합니다).

내장 `OpKind`(PLUS, MAX 등)는 int/float 같은 **스칼라 타입**에만 정의되어 있어, 구조체를
비교하려면(여기서는 "g 필드가 더 큰 쪽") 반드시 **커스텀 함수**가 필요합니다. `max_g`는 3절의
`add_even`과 같은 원리로 컴파일되며, 두 `Pixel` 레코드를 받아 하나를 그대로
반환하는 형태입니다 — 이런 패턴은 NumPy/CuPy만으로는 표현하기 까다로운 "필드 기준 argmax류"
연산을 병렬 리덕션으로 자연스럽게 확장한 예입니다.

In [ ]:
import numpy as np
import cupy as cp
from cuda.compute import gpu_struct, OpKind

@gpu_struct
class Pixel:
    r: np.int32
    g: np.int32
    b: np.int32

def max_g(x, y): 
    return x if x.g > y.g else y

# (10, 3) 모양의 RGB 난수 배열을 생성한 뒤, Pixel 구조체 배열로 뷰(View) 변환
d_rgb = cp.random.randint(0, 256, (10, 3), dtype=np.int32).view(Pixel.dtype)
d_out = cp.empty(1, dtype=Pixel.dtype)
h_init = np.zeros(1, dtype=Pixel.dtype)

cuda.compute.reduce_into(
    d_in=d_rgb, 
    d_out=d_out, 
    op=max_g, 
    num_items=d_rgb.size, 
    h_init=h_init
)

print('전체 배열 (원본):\n', d_rgb)
print('가장 초록색이 강한 픽셀:', d_out.get()[0])

### 7.3 종합 평가 — 센서 '직전 최고치' (ref: GTC CUDA python)

센서 온도 스트림에서 **각 시점 직전까지의 최고 온도**를 구하세요 — `max` 연산 **exclusive scan**으로.
`prev_peak[i] = max(readings[0..i-1])` (i=0은 초기값).

### 조금 더 구체적으로

이 문제("각 시점 직전까지의 최고치")는 **순차 코드로는 자명하지만 병렬화가 까다로운** 대표적인
패턴입니다. 순수 파이썬으로 짜면 `for i in range(1, N): prev_peak[i] = max(prev_peak[i-1],
readings[i-1])`처럼 이전 결과에 의존하는 루프가 되어 GPU 스레드에 그대로 나눠줄 수 없습니다
(00에서 강조한 "직렬 for 루프를 피하라"는 규칙이 정확히 이런 상황을 가리킵니다). `exclusive_scan`
+ 커스텀 `max_op`을 쓰면 이 의존 관계를 병렬 트리 알고리즘(7절에서 언급한 Hillis-Steele/Blelloch류)
으로 바꿔, `O(log N)` 단계 안에 100만 개를 동시에 처리합니다.

같은 패턴이 실무에서는 주가의 '지금까지의 고점(running maximum)', 센서 모니터링의 '누적 최고
온도/압력 경보', 게임의 '지금까지의 최고 점수' 등으로 반복해서 등장합니다 — `max`를 다른 결합
연산(`min`, 커스텀 비교자)으로 바꾸면 그대로 재사용할 수 있는 범용 패턴입니다.

In [ ]:
import numpy as np
import cupy as cp
import cuda.compute

# 커스텀 최댓값 연산 함수
def max_op(a, b): 
    return a if a > b else b

N = 1_000_000
readings = cp.random.randint(0, 100, size=N, dtype=np.int32)
prev_peak = cp.empty(N, dtype=np.int32)

# 1. 매우 작은 초기값 설정 (-2^31)
h_init = np.array([-2147483648], dtype=np.int32)

# 2. CCCL exclusive_scan 호출
cuda.compute.exclusive_scan(
    d_in=readings, 
    d_out=prev_peak, 
    op=max_op, 
    init_value=h_init,  
    num_items=N
)

# --- 3. 검증 (Validation - NumPy 사용) ---
# CuPy 대신 NumPy로 데이터를 가져와서 검증합니다.
# CuPy에서는 아직 maximum.accumulate 기능을 구현해 놓지 않았습니다.
readings_np = readings.get()

expected_np = np.empty(N, dtype=np.int32)
expected_np[0] = h_init[0]
expected_np[1:] = np.maximum.accumulate(readings_np[:-1])

# 작은 N(앞부분 10개) 결과 눈으로 확인하기
print("입력 데이터 :", readings_np[:10])
print("CCCL 결과   :", prev_peak[:10].get())
print("NumPy(기대값):", expected_np[:10])

# 전체 100만 개 데이터 일치 여부 검증
np.testing.assert_array_equal(prev_peak.get(), expected_np)
print("\n[전체 데이터 검증 완료] CCCL exclusive_scan == NumPy accumulate")

<a id="7"></a>
## 8. 연습 — 커스텀 op 최댓값

**연습 — 커스텀 op로 최댓값**: `reduce_into`에 `max_op(a,b)`와 적절한 `h_init`(아주 작은 값)을 줘서 최댓값을 구하세요.

### 조금 더 구체적으로

이 연습은 2절(`reduce_into` 기본 사용법)과 3절(커스텀 이항연산)을 그대로 합친 문제입니다. 핵심은
**항등원 선택**입니다 — `PLUS`의 항등원은 0이지만, `max`의 항등원은 "이 dtype에서 나올 수 있는
가장 작은 값"이어야 합니다. `int32` 범위를 벗어나는 값을 `h_init`으로 주면 오히려 잘못된 결과가
나올 수 있으니, dtype에 정확히 맞는 최솟값을 계산해서 넣는 것이 이 문제의 관건입니다(구체적인
계산 방법은 해답에서 확인하세요).

In [ ]:

import numpy as np
import cupy as cp
import cuda.compute

# from course_utils import allclose

def max_op(a, b):
    return a if a > b else b

# 100만 개의 무작위 정수 배열 생성 (0 ~ 999,999)
d_in = cp.random.randint(0, 10**6, size=1_000_000, dtype=np.int32)
d_out = cp.empty(1, dtype=np.int32)

# np.iinfo(np.int32).min 을 사용하여 int32의 가장 작은 값(-2147483648)을 동적으로 가져옵니다.
# 이전 예제에서는 -2147483648을 직접 쳤지만, 실무에서는 데이터 타입이 수시로 바뀔 수 있습니다. 
# np.iinfo(np.int32).min을 사용하면 데이터 타입에 맞는 최소값을 알아서 찾아주기 때문에 보다 안전한 코드입니다.
h_init = np.array([np.iinfo(np.int32).min], dtype=np.int32)

# reduce_into 호출
cuda.compute.reduce_into(
    d_in=d_in,
    d_out=d_out,
    op=max_op,
    num_items=d_in.size,
    h_init=h_init
)

# CuPy 내장 함수인 .max()와 결과 비교하기
print('CCCL 최댓값:', int(d_out[0]), '/ CuPy 최댓값:', int(d_in.max()))

# 검증 (두 값이 일치하면 'allclose OK' 출력)
allclose(int(d_in.max()), int(d_out[0]), name='reduce max')

### 체크포인트
- [ ] `reduce_into`로 합·커스텀 리덕션을 수행했다
- [ ] `unary_transform`·iterator를 사용했다
- [ ] cccl / Numba / CuPy의 사용 시점을 구분한다
- [ ] (참고) scan/sort 등 다른 알고리즘도 같은 방식으로 쓸 수 있음을 안다

다음: **`11_rawkernel`** — RawKernel(CUDA C)과 커널 개념 적용(최적화) 기술.